In [29]:
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, log_loss
import pandas as pd
import numpy as np


# Use the same prepared train/test split as RandomForest.ipynb.
X_train = pd.read_csv("../src/Dataset/X_train.csv")
X_test = pd.read_csv("../src/Dataset/X_test.csv").to_numpy()
y_train = pd.read_csv("../src/Dataset/y_train.csv").to_numpy().ravel()
y_test = pd.read_csv("../src/Dataset/y_test.csv").to_numpy().ravel()

display(X_train.head())
X_train.to_numpy()
#Train model. Keep the test set out of training and model selection.
# model = CatBoostClassifier(
#     iterations=1000,
#     loss_function="MultiClass",
#     random_seed=42,
#     verbose=False,
#     allow_writing_files=False,
# )

#model = CatBoostClassifier(iterations=500, loss_function='Logloss', verbose=0)

base_model = CatBoostClassifier(iterations=500, loss_function='MultiClass', verbose=0,auto_class_weights="Balanced")
base_model.fit(X_train, y_train)

# 2. Fit and calibrate using 5-fold CV
calibrated_model = CalibratedClassifierCV(
    estimator=CatBoostClassifier(iterations=500, loss_function='MultiClass', verbose=0),
    method='sigmoid',
    cv=5
)
calibrated_model.fit(X_train, y_train)

# 3. Get full probability matrices (shape: [n_samples, n_classes])
raw_probs = base_model.predict_proba(X_test)
cal_probs = calibrated_model.predict_proba(X_test)
# 1. Print numerical calibration metrics
print(f"Raw Brier Score:        {brier_score_loss(y_test, raw_probs):.4f}")
print(f"Calibrated Brier Score: {brier_score_loss(y_test, cal_probs):.4f}")
print("-" * 40)
print(f"Raw Log-Loss:           {log_loss(y_test, raw_probs):.4f}")
print(f"Calibrated Log-Loss:    {log_loss(y_test, cal_probs):.4f}")

classes = base_model.classes_

for idx, cls in enumerate(classes):
    print(f"=== Class: {cls} ===")
    df_cls = pd.DataFrame({
        'Is_Class': (y_test == cls).astype(int),
        'Raw_Prob': raw_probs[:, idx],
        'Calibrated_Prob': cal_probs[:, idx]
    })
    df_cls['Prob_Bin'] = pd.cut(df_cls['Calibrated_Prob'], bins=5)
    
    summary = df_cls.groupby('Prob_Bin', observed=False).agg(
        Sample_Count=('Is_Class', 'count'),
        Actual_Rate=('Is_Class', 'mean'),
        Mean_Raw_Prob=('Raw_Prob', 'mean'),
        Mean_Calibrated_Prob=('Calibrated_Prob', 'mean')
    )
    print(summary.round(4), "\n")
# CatBoost returns class predictions as a column; flatten for sklearn metrics.
y_pred = calibrated_model.predict(X_test).ravel()
print(classification_report(
    y_test, y_pred,
    labels=[0, 1, 2],
    target_names=['Healthy', 'Low Risk', 'Injured'],
    zero_division=0,
))




# cal_probs columns: [0: Healthy, 1: Low Risk, 2: Injured]
p_healthy = cal_probs[:, 0]
p_low_risk = cal_probs[:, 1]
p_injured = cal_probs[:, 2]

y_pred_custom = np.zeros(len(cal_probs), dtype=int)

# Flag Low Risk when combined non-healthy risk is high OR Low Risk exceeds base rate
y_pred_custom[(p_low_risk >= 0.21) | ((p_low_risk + p_injured) >= 0.42)] = 1

# Keep your strong Injured threshold on top
y_pred_custom[p_injured >= 0.25] = 2

print("=== Custom Threshold Classification Report ===")
print(classification_report(
    y_test,
    y_pred_custom,
    labels=[0, 1, 2],
    target_names=["Healthy", "Low Risk", "Injured"],
    zero_division=0,
))



,0,1,2,3,4,5
0,-0.762150,-0.422623,-0.815974,-0.578288,0.593522,-0.565252
1,-0.166016,0.342421,0.014358,0.283938,-0.646644,0.559957
2,0.227617,0.363722,-0.642939,0.219397,-0.926260,1.014962
3,-0.182313,-0.494727,0.922822,0.944759,-0.105616,1.677469
4,-1.922392,-1.626282,-0.032538,-0.131392,1.209695,0.044618


Raw Brier Score:        0.5654
Calibrated Brier Score: 0.4658
----------------------------------------
Raw Log-Loss:           0.8826
Calibrated Log-Loss:    0.7736
=== Class: 0 ===
                Sample_Count  Actual_Rate  Mean_Raw_Prob  Mean_Calibrated_Prob
Prob_Bin                                                                      
(0.168, 0.291]           198       0.2222         0.0555                0.2398
(0.291, 0.414]           222       0.3423         0.1218                0.3552
(0.414, 0.537]           366       0.4454         0.1888                0.4767
(0.537, 0.659]           668       0.5853         0.2888                0.6075
(0.659, 0.782]          3172       0.7210         0.5319                0.7236 

=== Class: 1 ===
                Sample_Count  Actual_Rate  Mean_Raw_Prob  Mean_Calibrated_Prob
Prob_Bin                                                                      
(0.13, 0.183]            601       0.1115         0.0913                0.1610
(0.183, 0

In [30]:
import numpy as np
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, f1_score, cohen_kappa_score

# 1. Get out-of-fold probabilities on X_train so we don't overfit X_test
oof_probs = cross_val_predict(calibrated_model, X_train, y_train, cv=5, method="predict_proba")
oof_risk = oof_probs[:, 1] * 1.0 + oof_probs[:, 2] * 2.0

best_score = -1
best_cutoffs = (0.45, 0.80)

# 2. Search for optimal Low Risk (c1) and Injured (c2) risk_score cutoffs
for c1 in np.arange(0.35, 0.65, 0.02):
    for c2 in np.arange(c1 + 0.15, 1.10, 0.02):
        preds = np.zeros(len(oof_risk), dtype=int)
        preds[oof_risk >= c1] = 1
        preds[oof_risk >= c2] = 2
        
        # Combine Macro F1 and Ordinal Kappa (QWK) for balanced ordinal performance
        macro_f1 = f1_score(y_train, preds, average="macro")
        qwk = cohen_kappa_score(y_train, preds, weights="quadratic")
        combined = 0.5 * macro_f1 + 0.5 * qwk
        
        if combined > best_score:
            best_score = combined
            best_cutoffs = (c1, c2)

c1_opt, c2_opt = best_cutoffs
print(f"Optimal Risk Cutoffs -> Low Risk: {c1_opt:.2f}, Injured: {c2_opt:.2f}\n")

# 3. Apply optimal cutoffs to your existing test risk_score
test_risk = cal_probs[:, 1] * 1.0 + cal_probs[:, 2] * 2.0
y_pred_final = np.zeros(len(test_risk), dtype=int)
y_pred_final[test_risk >= c1_opt] = 1
y_pred_final[test_risk >= c2_opt] = 2

print("=== Final Tuned Ordinal Classification Report ===")
print(classification_report(
    y_test,
    y_pred_final,
    labels=[0, 1, 2],
    target_names=["Healthy", "Low Risk", "Injured"],
    zero_division=0,
))
print(f"Quadratic Weighted Kappa (QWK): {cohen_kappa_score(y_test, y_pred_final, weights='quadratic'):.4f}")


Optimal Risk Cutoffs -> Low Risk: 0.43, Injured: 0.70

=== Final Tuned Ordinal Classification Report ===
              precision    recall  f1-score   support

     Healthy       0.73      0.76      0.74      2961
    Low Risk       0.17      0.13      0.15       971
     Injured       0.51      0.61      0.55       694

    accuracy                           0.60      4626
   macro avg       0.47      0.50      0.48      4626
weighted avg       0.58      0.60      0.59      4626

Quadratic Weighted Kappa (QWK): 0.4370


In [31]:
from sklearn.metrics import brier_score_loss, roc_auc_score

# Probability columns follow model.classes_: Healthy (0), Low Risk (1), Injured (2).
y_proba = model.predict_proba(X_test)

# Multiclass Brier score: sum of squared errors across classes, averaged over samples.
# Range: 0 to 2; lower is better.
brier = brier_score_loss(
    y_test, y_proba, labels=model.classes_, scale_by_half=False
)

# Treat each class as positive versus all other classes, then average equally.
# Higher is better; 0.5 indicates chance-level ranking.
roc_auc = roc_auc_score(
    y_test, y_proba, labels=model.classes_, multi_class="ovr", average="macro"
)

print(f"Multiclass Brier score (0-2, lower is better): {brier:.4f}")
print(f"Macro ROC AUC (one-vs-rest, higher is better): {roc_auc:.4f}")


Multiclass Brier score (0-2, lower is better): 0.4730
Macro ROC AUC (one-vs-rest, higher is better): 0.6980


In [6]:
from sklearn.metrics import log_loss

# predict_proba returns one probability per class for each test sample.
y_proba = model.predict_proba(X_test)
class_names = {0: "Healthy", 1: "Low Risk", 2: "Injured"}
probability_predictions = pd.DataFrame(
    y_proba,
    columns=[f"P({class_names[label]})" for label in model.classes_],
)
probability_predictions.insert(0, "Actual", pd.Series(y_test).map(class_names))
probability_predictions.insert(1, "Predicted", pd.Series(y_pred.ravel()).map(class_names))
print("Probability predictions (first 5 test samples):")
print(probability_predictions.head().to_string(index=False, float_format="{:.4f}".format))

# Evaluate each class against all other classes using probabilities, not hard labels.
rows = []
for column, label in enumerate(model.classes_):
    is_class = (y_test == label).astype(int)
    probabilities = y_proba[:, column]
    rows.append({
        "Class": class_names[label],
        "Support": int(is_class.sum()),
        "Actual proportion": is_class.mean(),
        "Mean predicted probability": probabilities.mean(),
        "Brier (0-1)": brier_score_loss(is_class, probabilities, scale_by_half=True),
        "ROC AUC (OvR)": roc_auc_score(is_class, probabilities),
    })
probability_report = pd.DataFrame(rows).set_index("Class")
print("\nPer-class probability report:")
print(probability_report.to_string(float_format="{:.4f}".format))
print(f"\nMulticlass log loss (lower is better): {log_loss(y_test, y_proba, labels=model.classes_):.4f}")
print("Brier: lower is better. ROC AUC: higher is better (0.5 is chance-level ranking).")
# The multiclass Brier score above equals the sum of the per-class Brier scores.


Probability predictions (first 5 test samples):
  Actual Predicted  P(Healthy)  P(Low Risk)  P(Injured)
 Injured   Healthy      0.5433       0.1600      0.2967
 Healthy   Injured      0.3238       0.0955      0.5807
Low Risk   Healthy      0.6716       0.3277      0.0007
 Injured   Healthy      0.6657       0.1258      0.2085
Low Risk   Healthy      0.5734       0.2964      0.1302

Per-class probability report:
          Support  Actual proportion  Mean predicted probability  Brier (0-1)  ROC AUC (OvR)
Class                                                                                       
Healthy      2961             0.6401                      0.6635       0.2170         0.6444
Low Risk      971             0.2099                      0.1936       0.1711         0.5462
Injured       694             0.1500                      0.1429       0.0903         0.8930

Multiclass log loss (lower is better): 0.7781
Brier: lower is better. ROC AUC: higher is better (0.5 is chance-level ra